# Phase 4 — Full Benchmark Analysis

Four feedback conditions (`vanilla`, `neutral`, `sage`, `combined_neutral`) compared head-to-head with `gemini-3-flash`, 10 seeds, 500 candidates per run, 20 MA-BBOB instances × 5 eval seeds.

**Analysis plan (Approach C, prior-targeted):**
1. Data loading & sanity checks
2. Failure rates
3. Headline: final best-AOCC comparison (4 conditions)
4. Convergence dynamics — curves, snapshots, budget-to-threshold, AUC-AOCC
5. Phase 3 → Phase 4 consistency check (does neutral scale?)
6. Per-instance / per-function-group breakdown
7. Behavioural profiles of generated algorithms
8. Algorithmic diversity (t-SNE + within-run pairwise distance)
9. Improvement burst analysis
10. Summary table

In [ ]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10})
sns.set_style('whitegrid')

RESULTS_DIR = Path('/Users/maxharell/repos/thesis/results_phase4')
FIG_DIR = Path('/Users/maxharell/repos/thesis/analysis/figs_phase4')
FIG_DIR.mkdir(exist_ok=True)

CONDITIONS = ['vanilla', 'neutral', 'sage', 'combined_neutral']
COND_LABELS = {'vanilla': 'Vanilla', 'neutral': 'Neutral (5 feat)',
               'sage': 'SAGE', 'combined_neutral': 'Neutral + SAGE'}
COND_COLORS = {'vanilla': '#888888', 'neutral': '#2E86AB',
               'sage': '#E63946', 'combined_neutral': '#6A4C93'}
SEEDS = list(range(10))
BUDGET = 500

# 20 MA-BBOB instances and their BBOB function-group mapping.
TRAINING_INSTANCES = [22, 93, 166, 196, 203, 288, 321, 408, 480, 513,
                      528, 598, 697, 781, 784, 803, 894, 947, 951, 999]

# 5 neutral features tracked in Phase 4
NEUTRAL_FEATURES = [
    'intensification_ratio',
    'dimension_convergence_heterogeneity',
    'fitness_plateau_fraction',
    'avg_improvement',
    'improvement_spatial_correlation',
]

## 1. Load all summary CSVs

One row per (condition, seed, generation). 4 × 10 × 500 = 20,000 rows.

In [ ]:
dfs = []
for cond in CONDITIONS:
    for seed in SEEDS:
        fp = RESULTS_DIR / cond / f'seed-{seed}' / 'summary.csv'
        d = pd.read_csv(fp)
        d['condition'] = cond
        d['seed'] = seed  # override the logged seed, which equals run seed anyway
        dfs.append(d)
df = pd.concat(dfs, ignore_index=True)

# Treat failures as NaN AOCC for best-so-far logic.
df['AOCC_valid'] = df['AOCC'].where(df['run_status'] == 'success')

# Best-so-far per (condition, seed)
df = df.sort_values(['condition', 'seed', 'generation']).reset_index(drop=True)
df['best_so_far'] = df.groupby(['condition', 'seed'])['AOCC_valid'].cummax()

print(df.shape)
df[['condition', 'seed', 'generation', 'algorithm_name', 'AOCC',
    'run_status', 'best_so_far']].head()

## 2. Failure rates

Does feedback format affect validity? Check whether more informative feedback leads the LLM to attempt more ambitious (failure-prone) or safer code.

In [ ]:
fail_tbl = (
    df.groupby('condition')
    .agg(n_total=('run_status', 'size'),
         n_fail=('run_status', lambda s: (s == 'failure').sum()))
    .assign(fail_pct=lambda x: 100 * x.n_fail / x.n_total)
    .loc[CONDITIONS]
)
print(fail_tbl)

# Per-seed failure rates (for distribution / CI)
per_seed_fail = (
    df.groupby(['condition', 'seed'])
    .agg(fail_pct=('run_status', lambda s: 100 * (s == 'failure').mean()))
    .reset_index()
)
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=per_seed_fail, x='condition', y='fail_pct', order=CONDITIONS,
            palette=[COND_COLORS[c] for c in CONDITIONS], ax=ax)
sns.stripplot(data=per_seed_fail, x='condition', y='fail_pct', order=CONDITIONS,
              color='black', size=3, alpha=0.6, ax=ax)
ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS], rotation=15)
ax.set_ylabel('Failure rate per seed (%)')
ax.set_xlabel('')
ax.set_title('Failure rate distribution across seeds')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_failure_rates.pdf', bbox_inches='tight')
plt.show()

# Kruskal-Wallis on per-seed failure rates
groups = [per_seed_fail[per_seed_fail.condition == c].fail_pct.values
          for c in CONDITIONS]
H, p = stats.kruskal(*groups)
print(f'\nKruskal-Wallis (failure rate across conditions): H={H:.2f}, p={p:.4f}')

## 3. Headline — final best-AOCC comparison

Mean best-AOCC at the end of the 500-candidate budget, per condition. Statistical tests: Kruskal–Wallis across four conditions, pairwise Mann–Whitney U with Holm correction, Cliff's delta for effect sizes, bootstrap 95% CIs.

In [ ]:
def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    n_gt = np.sum(a[:, None] > b[None, :])
    n_lt = np.sum(a[:, None] < b[None, :])
    return (n_gt - n_lt) / (len(a) * len(b))


def bootstrap_ci(x, n=10000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    boots = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return np.quantile(boots, [alpha / 2, 1 - alpha / 2])


# Final best-so-far per seed
final = (
    df.groupby(['condition', 'seed'])
    .agg(final_best=('best_so_far', 'last'))
    .reset_index()
)
summary = (
    final.groupby('condition')
    .agg(mean=('final_best', 'mean'), std=('final_best', 'std'),
         median=('final_best', 'median'), min=('final_best', 'min'),
         max=('final_best', 'max'))
    .loc[CONDITIONS]
)
summary['ci_lo'] = [bootstrap_ci(final[final.condition == c].final_best)[0] for c in CONDITIONS]
summary['ci_hi'] = [bootstrap_ci(final[final.condition == c].final_best)[1] for c in CONDITIONS]
print(summary.round(4))

In [ ]:
# Kruskal-Wallis across four conditions
groups = [final[final.condition == c].final_best.values for c in CONDITIONS]
H, p_kw = stats.kruskal(*groups)
print(f'Kruskal-Wallis: H={H:.3f}, p={p_kw:.4f}\n')

# Pairwise Mann-Whitney U with Holm correction
from itertools import combinations
pairs = list(combinations(CONDITIONS, 2))
raw_p, stats_out = [], []
for a, b in pairs:
    xa, xb = final[final.condition == a].final_best, final[final.condition == b].final_best
    u, pv = stats.mannwhitneyu(xa, xb, alternative='two-sided')
    d = cliffs_delta(xa, xb)
    raw_p.append(pv)
    stats_out.append((a, b, u, pv, d))

# Holm-Bonferroni
order = np.argsort(raw_p)
m = len(raw_p)
holm = np.empty(m)
for rank, idx in enumerate(order):
    holm[idx] = min(1.0, raw_p[idx] * (m - rank))
# enforce monotonicity
for i in range(1, m):
    idx_prev, idx_cur = order[i - 1], order[i]
    holm[idx_cur] = max(holm[idx_cur], holm[idx_prev])

pair_tbl = pd.DataFrame({
    'pair': [f'{a} vs {b}' for a, b, *_ in stats_out],
    'U': [u for _, _, u, _, _ in stats_out],
    'p_raw': [pv for _, _, _, pv, _ in stats_out],
    'p_holm': holm,
    'cliffs_delta': [d for _, _, _, _, d in stats_out],
}).round(4)
print(pair_tbl)

In [ ]:
# Boxplot + strip plot for final AOCC
fig, ax = plt.subplots(figsize=(6.5, 4))
sns.boxplot(data=final, x='condition', y='final_best', order=CONDITIONS,
            palette=[COND_COLORS[c] for c in CONDITIONS], ax=ax, width=0.55,
            fliersize=0)
sns.stripplot(data=final, x='condition', y='final_best', order=CONDITIONS,
              color='black', size=4, alpha=0.75, ax=ax, jitter=0.15)
for i, c in enumerate(CONDITIONS):
    m = summary.loc[c, 'mean']
    ax.plot([i - 0.22, i + 0.22], [m, m], color='red', lw=2, zorder=5)
ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS], rotation=15)
ax.set_ylabel('Final best-so-far AOCC')
ax.set_xlabel('')
ax.set_title(f'Final AOCC after 500 candidates (10 seeds, H={H:.2f}, p={p_kw:.4f})')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_final_aocc_boxplot.pdf', bbox_inches='tight')
plt.show()

## 4. Convergence dynamics

Does behavioural feedback help convergence *speed*, even if the ceiling is similar? Four views:
- Best-so-far curves (mean ± std across seeds)
- Snapshot AOCC at generations 50, 100, 250, 500
- Budget-to-threshold — how many generations to reach vanilla's median final AOCC
- AUC-AOCC — integrated best-so-far over the 500-candidate budget

In [ ]:
# Mean best-so-far trajectory (across seeds) per condition
curves = (
    df.groupby(['condition', 'generation'])['best_so_far']
    .agg(mean='mean', std='std', med='median')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
for c in CONDITIONS:
    sub = curves[curves.condition == c]
    ax.plot(sub.generation, sub['mean'], label=COND_LABELS[c], color=COND_COLORS[c], lw=1.8)
    ax.fill_between(sub.generation, sub['mean'] - sub['std'], sub['mean'] + sub['std'],
                    color=COND_COLORS[c], alpha=0.15)
ax.set_xlabel('Generation (candidate evaluation)')
ax.set_ylabel('Best-so-far AOCC (mean ± std across 10 seeds)')
ax.set_title('Convergence dynamics')
ax.legend(loc='lower right', frameon=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_convergence.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Snapshot AOCC at key generations
snapshots = [49, 99, 249, 499]  # 0-indexed: gens 50, 100, 250, 500
snap = (
    df[df.generation.isin(snapshots)]
    .groupby(['condition', 'generation'])['best_so_far']
    .agg(['mean', 'std'])
    .reset_index()
)
snap['label'] = snap.apply(lambda r: f"{r['mean']:.3f}±{r['std']:.3f}", axis=1)
snap_pivot = snap.pivot(index='condition', columns='generation', values='label').loc[CONDITIONS]
snap_pivot.columns = [f'gen {g+1}' for g in snap_pivot.columns]
print('Snapshot best-so-far AOCC (mean ± std across seeds):')
print(snap_pivot.to_string())

# Significance at each snapshot
print('\nKruskal-Wallis at each snapshot:')
for g in snapshots:
    gs = [df[(df.condition == c) & (df.generation == g)].best_so_far.values for c in CONDITIONS]
    H, p = stats.kruskal(*gs)
    print(f'  gen {g+1:3d}: H={H:.2f}, p={p:.4f}')

In [ ]:
# Budget-to-threshold: generations needed to reach vanilla's median final AOCC
threshold = final[final.condition == 'vanilla'].final_best.median()
print(f'Threshold (vanilla median final AOCC): {threshold:.4f}\n')

def first_reach(series, thr):
    mask = series >= thr
    if not mask.any():
        return np.nan  # never reached
    return int(mask.idxmax() - series.index[0]) + 1  # 1-indexed generation count

budget_to_thr = (
    df.groupby(['condition', 'seed'])['best_so_far']
    .apply(lambda s: first_reach(s, threshold))
    .reset_index(name='gens_to_threshold')
)
thr_summary = (
    budget_to_thr.groupby('condition')['gens_to_threshold']
    .agg(['mean', 'median', 'std',
          ('n_reached', lambda x: x.notna().sum()),
          ('n_total', 'size')])
    .loc[CONDITIONS]
)
print('Budget-to-threshold (gens to reach vanilla-median final AOCC):')
print(thr_summary.round(1))

# Plot
fig, ax = plt.subplots(figsize=(6, 3.5))
for c in CONDITIONS:
    vals = budget_to_thr[budget_to_thr.condition == c].gens_to_threshold.dropna()
    ax.scatter([c] * len(vals), vals, color=COND_COLORS[c], s=45, alpha=0.8)
    if len(vals):
        ax.plot([c, c], [vals.median(), vals.median()], marker='_', color='red', mew=2, ms=30)
ax.axhline(BUDGET, color='grey', linestyle='--', lw=0.8, label='Max budget (500)')
ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS], rotation=15)
ax.set_ylabel(f'Gens to reach AOCC ≥ {threshold:.3f}')
ax.set_title('Convergence speed — lower is faster')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_budget_to_threshold.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# AUC-AOCC: integrated best-so-far over 500-candidate budget.
# Higher = better anytime performance (fast convergence + high ceiling).
auc = (
    df.groupby(['condition', 'seed'])['best_so_far']
    .apply(lambda s: float(np.trapezoid(s.fillna(0).values) / BUDGET))
    .reset_index(name='auc_aocc')
)
auc_summary = (
    auc.groupby('condition')['auc_aocc']
    .agg(['mean', 'std', 'median']).loc[CONDITIONS]
)
print('AUC-AOCC (integrated best-so-far / budget):')
print(auc_summary.round(4))

groups = [auc[auc.condition == c].auc_aocc.values for c in CONDITIONS]
H, p = stats.kruskal(*groups)
print(f'\nKruskal-Wallis on AUC-AOCC: H={H:.3f}, p={p:.4f}')

# Pairwise
for a, b in combinations(CONDITIONS, 2):
    xa, xb = auc[auc.condition == a].auc_aocc, auc[auc.condition == b].auc_aocc
    _, pv = stats.mannwhitneyu(xa, xb, alternative='two-sided')
    d = cliffs_delta(xa, xb)
    print(f'  {a:20s} vs {b:20s}  p={pv:.4f}  δ={d:+.2f}')

## 5. Phase 3 → Phase 4 consistency check

Phase 3 result (see Ch 5): neutral beats directional/comparative at n=5 seeds, 100 candidates, one feature at a time. Best single-feature neutral was `intensification_ratio` at **0.907**. Overall neutral mean across 10 single-feature conditions was **0.869**.

Phase 4 neutral uses *all five* top-ranked neutral features bundled, at 500 candidates and 10 seeds. Does the signal hold when scaled up?

In [ ]:
# Phase 3 reference (from the thesis results chapter, Table 5.X / steering-detail)
phase3_ref = {
    'vanilla_baseline': 0.862,                # Phase 3 vanilla (100 cand, 5 seeds)
    'neutral_best_single': 0.907,             # best single-feature neutral (intens_ratio)
    'neutral_mean_single': 0.869,             # mean across all 10 neutral conditions
}
phase4_vals = {c: final[final.condition == c].final_best.mean() for c in CONDITIONS}

# Compare on the AOCC scale the thesis already uses.
print('Phase 3 (100 cand, 5 seeds, 10 MA-BBOB instances):')
for k, v in phase3_ref.items():
    print(f'  {k:25s}: {v:.4f}')
print()
print('Phase 4 (500 cand, 10 seeds, 20 MA-BBOB instances):')
for c in CONDITIONS:
    print(f'  {c:25s}: {phase4_vals[c]:.4f}')

# Decompose the delta. Phase 3 neutral beat vanilla by +0.045 on the single best feature.
# Phase 4 asks: does bundled-5 neutral beat vanilla at n=10, 500 cand?
delta_p3 = phase3_ref['neutral_best_single'] - phase3_ref['vanilla_baseline']
delta_p4 = phase4_vals['neutral'] - phase4_vals['vanilla']
print(f'\nΔ(neutral − vanilla) Phase 3 (single best feat, n=5):  {delta_p3:+.4f}')
print(f'Δ(neutral − vanilla) Phase 4 (bundled 5 feat, n=10):  {delta_p4:+.4f}')

# Caveat: Phase 3 and Phase 4 differ in instances (10 vs 20), budget (100 vs 500), seeds.
# So this is a directional, not apples-to-apples, check.

## 6. Per-instance / per-function-group breakdown

The 20 training instances are drawn from 5 BBOB function groups (separable, low-moderate conditioning, high/unimodal, multi-adequate global structure, multi-weak global structure). Load per-instance AUCs from `log.jsonl` and aggregate.

In [ ]:
# Load per-instance AUCs for the final best-so-far algorithm per (condition, seed).
# Strategy: for each (condition, seed), find the generation with the best AOCC,
# then pull that row's metadata.aucs array from log.jsonl.

def load_logs(condition, seed):
    seed_dir = RESULTS_DIR / condition / f'seed-{seed}'
    rows = []
    for log in sorted(seed_dir.glob('run-*/log.jsonl')):
        with open(log) as f:
            for line in f:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows


def meta_aucs(row):
    m = row.get('metadata')
    if isinstance(m, str):
        try:
            m = ast.literal_eval(m)
        except (ValueError, SyntaxError):
            return None
    if not isinstance(m, dict):
        return None
    return m.get('aucs')


# Collect per-instance AUCs for each candidate (not just the best) — used later too.
per_cand = []
for cond in CONDITIONS:
    for seed in SEEDS:
        logs = load_logs(cond, seed)
        for row in logs:
            aucs = meta_aucs(row)
            if aucs is None or len(aucs) != 100:
                continue
            arr = np.asarray(aucs).reshape(20, 5)  # 20 instances × 5 eval seeds
            per_cand.append({
                'condition': cond,
                'seed': seed,
                'generation': int(row.get('generation', -1)),
                'aocc': float(row.get('fitness', np.nan)),
                'per_instance': arr.mean(axis=1),  # mean across 5 eval seeds per instance
            })
print(f'Loaded per-instance scores for {len(per_cand)} candidates')

In [ ]:
# For each (condition, seed), take the final best-so-far candidate's per-instance AUC.
pc_df = pd.DataFrame(per_cand)
pc_df = pc_df.sort_values(['condition', 'seed', 'generation']).reset_index(drop=True)

# Best-so-far index per (condition, seed)
def pick_best(group):
    valid = group.dropna(subset=['aocc'])
    if valid.empty:
        return None
    idx = valid['aocc'].idxmax()
    return valid.loc[idx]

best_rows = (
    pc_df.groupby(['condition', 'seed'], group_keys=False).apply(pick_best).reset_index(drop=True)
)

# Stack per-instance AUCs into a (40, 20) matrix: 40 runs × 20 instances
inst_matrix = np.stack(best_rows['per_instance'].values)
print('Best-algorithm per-instance matrix shape:', inst_matrix.shape)

inst_df = pd.DataFrame(inst_matrix, columns=[f'inst_{i}' for i in TRAINING_INSTANCES])
inst_df.insert(0, 'condition', best_rows['condition'].values)
inst_df.insert(1, 'seed', best_rows['seed'].values)

# Mean per-instance AOCC by condition
cond_inst = inst_df.groupby('condition')[inst_df.columns[2:]].mean().loc[CONDITIONS]
print('\nPer-instance mean AOCC of best-found algorithm (across 10 seeds):')
print(cond_inst.round(3))

In [ ]:
# Per-instance heatmap.
fig, ax = plt.subplots(figsize=(12, 3.5))
sns.heatmap(cond_inst, annot=True, fmt='.2f', cmap='viridis',
            cbar_kws={'label': 'Mean best-found AOCC'}, ax=ax,
            yticklabels=[COND_LABELS[c] for c in CONDITIONS],
            xticklabels=[f'i{i}' for i in TRAINING_INSTANCES],
            annot_kws={'size': 7})
ax.set_title('Per-instance performance of best-found algorithm (mean across 10 seeds)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_per_instance_heatmap.pdf', bbox_inches='tight')
plt.show()

# Wins table: for each instance, which condition has highest mean?
winner = cond_inst.idxmax(axis=0)
win_counts = winner.value_counts().reindex(CONDITIONS, fill_value=0)
print('Instance-level wins:')
print(win_counts)

## 7. Behavioural profiles of generated algorithms

Do the four conditions produce behaviourally different algorithms, or do they converge to similar profiles? Track the 5 neutral features that are actually fed back in the `neutral` / `combined_neutral` conditions.

In [ ]:
# Use only valid candidates for behavioural analysis
bm_cols = [c for c in df.columns if c.startswith('bm_')]
valid = df[df.run_status == 'success'].copy()

tracked_bm = [f'bm_{f}' for f in NEUTRAL_FEATURES]
prof = valid[['condition'] + tracked_bm].dropna()

# Median per condition
prof_med = prof.groupby('condition')[tracked_bm].median().loc[CONDITIONS]
prof_med.columns = NEUTRAL_FEATURES
print('Median of each tracked neutral feature, per condition (all valid candidates):')
print(prof_med.round(3))

In [ ]:
# Violin plots — one per feature.
fig, axes = plt.subplots(1, len(NEUTRAL_FEATURES), figsize=(15, 3.5), sharey=False)
for ax, feat in zip(axes, NEUTRAL_FEATURES):
    col = f'bm_{feat}'
    d = valid[['condition', col]].dropna()
    # Clip extreme outliers for readability (per feature, 1st/99th percentile)
    lo, hi = d[col].quantile([0.01, 0.99])
    d = d[(d[col] >= lo) & (d[col] <= hi)]
    sns.violinplot(data=d, x='condition', y=col, order=CONDITIONS, ax=ax,
                   palette=[COND_COLORS[c] for c in CONDITIONS], inner='quartile',
                   cut=0)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels([COND_LABELS[c].split()[0] for c in CONDITIONS], rotation=30, fontsize=8)
plt.suptitle('Distribution of tracked behavioural features across conditions (valid candidates)',
             y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_behavioural_profiles.pdf', bbox_inches='tight')
plt.show()

## 8. Algorithmic diversity — the "escape" test

If behavioural feedback helps the LLM escape suboptimal code structures, we'd expect neutral / combined_neutral to produce more *diverse* algorithms — broader exploration in behaviour space.

Two cuts:
1. **t-SNE projection** of all valid candidates, coloured by condition.
2. **Within-seed mean pairwise behavioural distance** — does neutral spread wider?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

# All 32 behavioural features for diversity analysis.
bm_full = valid[bm_cols].replace([np.inf, -np.inf], np.nan).dropna()
ix_valid = bm_full.index
X = bm_full.values
# Winsorize to 1/99 per column then z-score.
lo, hi = np.nanquantile(X, [0.01, 0.99], axis=0)
Xw = np.clip(X, lo, hi)
Xs = StandardScaler().fit_transform(Xw)

print(f'Valid candidate matrix for t-SNE: {Xs.shape}')

# t-SNE (subsample if large for speed; here n≈17–20k is fine but slow — subsample to 6k if needed)
rng = np.random.default_rng(0)
if Xs.shape[0] > 6000:
    sub = rng.choice(Xs.shape[0], 6000, replace=False)
    Xs_sub = Xs[sub]
    cond_sub = valid.loc[ix_valid, 'condition'].values[sub]
else:
    Xs_sub = Xs
    cond_sub = valid.loc[ix_valid, 'condition'].values

tsne = TSNE(n_components=2, perplexity=40, init='pca', learning_rate='auto',
            random_state=0, n_iter=1000)
emb = tsne.fit_transform(Xs_sub)

fig, ax = plt.subplots(figsize=(7, 6))
for c in CONDITIONS:
    m = cond_sub == c
    ax.scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.35, label=COND_LABELS[c],
               color=COND_COLORS[c])
ax.set_title('t-SNE of behavioural feature space, coloured by condition')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(markerscale=2, loc='best', frameon=True)
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_tsne_conditions.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Within-seed mean pairwise behavioural distance.
# For each (condition, seed), compute mean pairwise Euclidean distance between z-scored
# behavioural vectors of valid candidates in that run.
valid2 = valid.dropna(subset=bm_cols).copy()
# Use global z-score (same scaler) for comparability across conditions.
scaler = StandardScaler().fit(valid2[bm_cols].clip(
    lower=pd.Series(lo, index=bm_cols),
    upper=pd.Series(hi, index=bm_cols),
    axis=1).values)
Zall = scaler.transform(valid2[bm_cols].values)
valid2['_zi'] = range(len(valid2))

rows = []
for (cond, seed), g in valid2.groupby(['condition', 'seed']):
    if len(g) < 3:
        continue
    Z = Zall[g['_zi'].values]
    # Mean pairwise distance (efficient)
    from scipy.spatial.distance import pdist
    d = pdist(Z, metric='euclidean')
    rows.append({'condition': cond, 'seed': seed, 'mean_pairwise_dist': float(d.mean()),
                 'n_valid': len(g)})
div_df = pd.DataFrame(rows)
print(div_df.groupby('condition')['mean_pairwise_dist'].agg(['mean', 'std', 'median']).loc[CONDITIONS].round(3))

groups = [div_df[div_df.condition == c]['mean_pairwise_dist'].values for c in CONDITIONS]
H, p = stats.kruskal(*groups)
print(f'\nKruskal-Wallis (diversity across conditions): H={H:.2f}, p={p:.4f}')

fig, ax = plt.subplots(figsize=(5.5, 3.5))
sns.boxplot(data=div_df, x='condition', y='mean_pairwise_dist', order=CONDITIONS,
            palette=[COND_COLORS[c] for c in CONDITIONS], ax=ax, width=0.55, fliersize=0)
sns.stripplot(data=div_df, x='condition', y='mean_pairwise_dist', order=CONDITIONS,
              color='black', size=3, alpha=0.7, ax=ax)
ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS], rotation=15)
ax.set_ylabel('Mean pairwise behavioural distance (within run)')
ax.set_xlabel('')
ax.set_title('Behavioural-space exploration per seed')
plt.tight_layout()
plt.savefig(FIG_DIR / 'p4_diversity_within_run.pdf', bbox_inches='tight')
plt.show()

## 9. Improvement burst analysis

Does neutral feedback produce more frequent (even if smaller) improvements in the best-so-far? Count distinct improvement events per run and their sizes. This speaks to whether behavioural feedback helps the LLM escape local plateaus by suggesting novel directions.

In [ ]:
def improvement_stats(bsf):
    s = bsf.ffill().bfill().values  # fill leading NaNs; best-so-far is non-decreasing
    deltas = np.diff(s)
    events = deltas[deltas > 1e-6]
    return pd.Series({
        'n_improvements': int(len(events)),
        'mean_improvement_size': float(events.mean()) if len(events) else 0.0,
        'max_improvement_size': float(events.max()) if len(events) else 0.0,
        'late_improvements': int((np.flatnonzero(deltas > 1e-6) > BUDGET * 0.5).sum()),
    })

burst = (
    df.groupby(['condition', 'seed'])['best_so_far']
    .apply(improvement_stats).unstack().reset_index()
)
print('Improvement events per run:')
print(burst.groupby('condition').agg(
    n_imp_mean=('n_improvements', 'mean'),
    n_imp_std=('n_improvements', 'std'),
    late_imp_mean=('late_improvements', 'mean'),
    imp_size_mean=('mean_improvement_size', 'mean'),
).loc[CONDITIONS].round(2))

# Test: more late-budget improvements → more escape behaviour
groups = [burst[burst.condition == c]['late_improvements'].values for c in CONDITIONS]
H, p = stats.kruskal(*groups)
print(f'\nKruskal-Wallis on late-budget improvements (gen > 250): H={H:.2f}, p={p:.4f}')

## 10. Summary table — all key metrics at a glance

One row per condition, consolidating the headline numbers used in the thesis write-up.

In [ ]:
summary_tbl = pd.DataFrame(index=CONDITIONS)
summary_tbl['Final AOCC mean'] = [final[final.condition == c].final_best.mean() for c in CONDITIONS]
summary_tbl['Final AOCC std'] = [final[final.condition == c].final_best.std() for c in CONDITIONS]
summary_tbl['AUC-AOCC'] = [auc[auc.condition == c].auc_aocc.mean() for c in CONDITIONS]
summary_tbl['Gens-to-thr (median)'] = [
    budget_to_thr[budget_to_thr.condition == c].gens_to_threshold.median() for c in CONDITIONS
]
summary_tbl['n reached thr'] = [
    int(budget_to_thr[budget_to_thr.condition == c].gens_to_threshold.notna().sum())
    for c in CONDITIONS
]
summary_tbl['Fail %'] = [fail_tbl.loc[c, 'fail_pct'] for c in CONDITIONS]
summary_tbl['Late improvements (mean)'] = [
    burst[burst.condition == c].late_improvements.mean() for c in CONDITIONS
]
summary_tbl['Within-run diversity'] = [
    div_df[div_df.condition == c].mean_pairwise_dist.mean() for c in CONDITIONS
]

print(summary_tbl.round(3).to_string())
summary_tbl.round(4).to_csv(FIG_DIR / 'p4_summary_table.csv')
print(f'\nSaved to {FIG_DIR / "p4_summary_table.csv"}')

### Interpretation checklist

When reading the output tables, look for:

- **Headline:** Does `neutral` or `combined_neutral` beat `vanilla` on final AOCC? Is it significant (Holm-adjusted)?
- **SAGE baseline:** Does `sage` match the LLaMEA-SAGE-literature gain over vanilla? This anchors whether the apparatus is working at all.
- **Speed vs ceiling:** If neutral's final AOCC is similar to vanilla's but its AUC-AOCC is higher or its gens-to-threshold is lower, that supports the "faster convergence" prior.
- **Combined synergy:** Does `combined_neutral` exceed max(neutral, sage), or does it come in between or below? Additive = synergy; between = no synergy; below = interference.
- **Diversity:** If behavioural feedback helps escape bad code structures, neutral should show higher within-run pairwise distance and/or more late-budget improvements.
- **Phase 3 → 4 consistency:** If Δ(neutral − vanilla) in Phase 4 is close to the Phase 3 single-feature result (+0.045), that tells you bundling features preserves the signal. If it shrinks to zero, bundling dilutes or the smaller-scale screening was noisy.